# Tarim Arazisi Siniri Segmentation Modeli

Bu yardimci model her pikseli `field` veya `non_field` olarak siniflandirir.
Calistirmadan once **Runtime > Change runtime type > T4 GPU** secin.

In [ ]:
import torch
print('CUDA aktif:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'GPU bulunamadi')

Sol taraftaki Files panelinden `field_segmentation.zip` dosyasini yukleyin. Yukleme tamamlanmadan sonraki hucreyi calistirmayin.

In [ ]:
!ls -lh field_segmentation.zip
!unzip -t field_segmentation.zip | tail -n 3
!unzip -q field_segmentation.zip -d /content/dataset

In [ ]:
from pathlib import Path
import numpy as np
from PIL import Image
import torch
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

ROOT = Path('/content/dataset/field_segmentation')
normalize = transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))

class FieldDataset(Dataset):
    def __init__(self, split):
        self.images = sorted((ROOT / 'images' / split).glob('*.jpg'))
        self.masks_dir = ROOT / 'masks' / split

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        image_path = self.images[index]
        image = np.asarray(Image.open(image_path).convert('RGB'), dtype=np.float32) / 255.0
        mask = np.asarray(Image.open(self.masks_dir / f'{image_path.stem}.png').convert('L')) > 0
        image = normalize(torch.from_numpy(image).permute(2, 0, 1))
        mask = torch.from_numpy(mask.astype(np.float32)).unsqueeze(0)
        return image, mask

train_loader = DataLoader(FieldDataset('train'), batch_size=8, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(FieldDataset('val'), batch_size=8, shuffle=False, num_workers=2, pin_memory=True)
print('train:', len(train_loader.dataset), 'val:', len(val_loader.dataset))

In [ ]:
from torchvision.models.segmentation import LRASPP_MobileNet_V3_Large_Weights, lraspp_mobilenet_v3_large
from torch import nn

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = lraspp_mobilenet_v3_large(weights=LRASPP_MobileNet_V3_Large_Weights.DEFAULT)
model.classifier.low_classifier = nn.Conv2d(40, 1, kernel_size=1)
model.classifier.high_classifier = nn.Conv2d(128, 1, kernel_size=1)
model = model.to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scaler = torch.amp.GradScaler('cuda', enabled=torch.cuda.is_available())

In [ ]:
def validate():
    model.eval()
    intersection = union = 0
    with torch.no_grad():
        for images, masks in val_loader:
            images, masks = images.to(device), masks.to(device)
            predictions = model(images)['out'].sigmoid() > 0.5
            targets = masks > 0.5
            intersection += (predictions & targets).sum().item()
            union += (predictions | targets).sum().item()
    return intersection / max(union, 1)

best_iou = 0.0
for epoch in range(1, 21):
    model.train()
    running_loss = 0.0
    for images, masks in train_loader:
        images, masks = images.to(device), masks.to(device)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
            loss = criterion(model(images)['out'], masks)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item()
    iou = validate()
    print(f'Epoch {epoch:02d} | loss={running_loss / len(train_loader):.4f} | val_iou={iou:.4f}')
    if iou > best_iou:
        best_iou = iou
        torch.save(model.state_dict(), '/content/field_boundary_model.pt')

print('En iyi IoU:', round(best_iou, 4))

In [ ]:
from google.colab import files
files.download('/content/field_boundary_model.pt')